# Finance Exam Assistant — QLoRA Fine-Tuning

**Base model:** `Qwen/Qwen2.5-1.5B-Instruct`  
**Dataset:** `MyDrive/data_FT/finexam_train_1000.jsonl`  
**Method:** supervised fine-tuning (SFT) with 4-bit QLoRA  
**Target:** an educational finance-exam assistant for RBI/SEBI/banking and financial-literacy questions.

Run in Google Colab with a GPU runtime (T4 or better). This notebook assumes each JSONL row uses the Qwen chat schema:

```json
{"messages":[
  {"role":"system","content":"..."},
  {"role":"user","content":"..."},
  {"role":"assistant","content":"..."}
]}
```

If your dataset uses a different schema, inspect the first record and adapt the validation/formatting cells before training. Keep a separate test set to evaluate generalization.

## 1. Mount Drive and configure paths

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DATA_PATH = Path("/content/drive/MyDrive/data_FT/finexam_train_1000.jsonl")
OUTPUT_DIR = Path("/content/drive/MyDrive/finexam_qlora_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Dataset exists:", DATA_PATH.exists())
print("Dataset:", DATA_PATH)
print("Output directory:", OUTPUT_DIR)
assert DATA_PATH.exists(), f"Dataset not found at {DATA_PATH}. Check the folder/file name in Drive."

Mounted at /content/drive
Dataset exists: True
Dataset: /content/drive/MyDrive/data_FT/finexam_train_1000.jsonl
Output directory: /content/drive/MyDrive/finexam_qlora_outputs


## 2. Load JSONL and inspect examples

In [2]:
import json

records = []
with DATA_PATH.open("r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSON on line {line_number}: {exc}") from exc

print("Number of records:", len(records))
assert records, "Dataset is empty."
print(json.dumps(records[0], indent=2, ensure_ascii=False))

Number of records: 1000
{
  "id": "fin-p2forms-tvm-003",
  "topic": "tvm",
  "subtopic": "present_value",
  "question_type": "mcq_numerical",
  "difficulty": "medium",
  "messages": [
    {
      "role": "user",
      "content": "Answer this SEBI Grade A Finance multiple-choice question. Give the best option and explain why.\n\nA sum will grow to Rs 62,500 after 4 years at 12% compounded annually. The amount invested today was closest to:\n\nOptions:\nA. Rs 62,500\nB. Rs 39,720\nC. Rs 26,838\nD. Rs 41,309\nE. Rs 42,230"
    },
    {
      "role": "assistant",
      "content": "Answer: B. Rs 39,720\n\nExplanation: Reversing the compounding: PV = 62,500/(1+0.12)^4 = Rs 39,720.\n\nWorked steps:\n1. (1+0.12)^4 = 1.57352\n2. PV = Rs 39,720"
    }
  ]
}


## 3. Validate the chat schema and check duplicates

In [3]:
from collections import Counter

for i, record in enumerate(records):
    assert isinstance(record, dict), f"Record {i} is not a JSON object."
    assert "messages" in record, (
        f"Record {i} has no 'messages' field. Adapt this notebook to your actual schema."
    )
    messages = record["messages"]
    assert isinstance(messages, list) and len(messages) >= 2, (
        f"Record {i} must contain a list of at least two messages."
    )
    assert messages[-1].get("role") == "assistant", (
        f"Record {i}: final message should be the assistant answer."
    )
    for j, message in enumerate(messages):
        assert message.get("role") in {"system", "user", "assistant"}, (
            f"Record {i}, message {j}: unexpected role {message.get('role')!r}."
        )
        assert isinstance(message.get("content"), str) and message["content"].strip(), (
            f"Record {i}, message {j}: content is empty or not text."
        )

serialized = [json.dumps(r, sort_keys=True, ensure_ascii=False) for r in records]
counts = Counter(serialized)
print("Total records:", len(records))
print("Unique records:", len(counts))
print("Repeated copies:", sum(n - 1 for n in counts.values() if n > 1))
print("Schema validation passed.")

Total records: 1000
Unique records: 1000
Repeated copies: 0
Schema validation passed.


## 4. Install compatible training libraries

In [4]:
%pip install -q \
    "transformers==4.46.3" \
    "trl==0.12.2" \
    "peft==0.13.2" \
    "accelerate==1.1.1" \
    "datasets==3.1.0" \
    "bitsandbytes==0.48.2" \
    "scikit-learn"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.9.0 which is incompatible.


In [5]:
import torch, transformers, trl, peft, bitsandbytes
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("No GPU detected. In Colab, select Runtime > Change runtime type > T4 GPU.")

PyTorch: 2.11.0+cu130
Transformers: 4.46.3
TRL: 0.12.2
PEFT: 0.13.2
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


## 5. Split into train, validation, and test sets

In [6]:
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# 80% train, 10% validation, 10% held-out test. For 1,000 records: 800/100/100.
train_val_records, test_records = train_test_split(
    records, test_size=0.10, random_state=42, shuffle=True
)
train_records, val_records = train_test_split(
    train_val_records, test_size=1/9, random_state=42, shuffle=True
)

dataset = DatasetDict({
    "train": Dataset.from_list(train_records),
    "validation": Dataset.from_list(val_records),
    "test": Dataset.from_list(test_records),
})
print(dataset)
print({split: len(dataset[split]) for split in dataset})
dataset.save_to_disk(str(OUTPUT_DIR / "dataset_split"))

DatasetDict({
    train: Dataset({
        features: ['id', 'topic', 'subtopic', 'question_type', 'difficulty', 'messages'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['id', 'topic', 'subtopic', 'question_type', 'difficulty', 'messages'],
        num_rows: 100
    })
    test: Dataset({
        features: ['id', 'topic', 'subtopic', 'question_type', 'difficulty', 'messages'],
        num_rows: 100
    })
})
{'train': 800, 'validation': 100, 'test': 100}


Saving the dataset (0/1 shards):   0%|          | 0/800 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

## 6. Load tokenizer, apply Qwen chat template, and inspect token lengths

In [7]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_messages(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

formatted = {
    split: dataset[split].map(format_messages)
    for split in ["train", "validation", "test"]
}
lengths = [
    len(tokenizer(row["text"], add_special_tokens=False)["input_ids"])
    for split in formatted.values()
    for row in split
]
print("Token lengths — min/median/p95/max:",
      min(lengths), int(np.percentile(lengths, 50)),
      int(np.percentile(lengths, 95)), max(lengths))
for limit in [512, 768, 1024, 1536]:
    print(f"Examples longer than {limit} tokens: {sum(n > limit for n in lengths)}/{len(lengths)}")

MAX_SEQ_LENGTH = 1024  # Adjust after reviewing the length statistics and available GPU memory.
print("\nExample formatted conversation:\n")
print(formatted["train"][0]["text"])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Token lengths — min/median/p95/max: 96 193 284 391
Examples longer than 512 tokens: 0/1000
Examples longer than 768 tokens: 0/1000
Examples longer than 1024 tokens: 0/1000
Examples longer than 1536 tokens: 0/1000

Example formatted conversation:

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Answer this SEBI Grade A Finance multiple-choice question. Give the best option and explain why.

A bank has NDTL of Rs 6,000 crore. At a CRR of 4.0% (as stated), the cash maintained with RBI is:

Options:
A. Rs 340 crore
B. Rs 240 crore
C. Rs 120 crore
D. Rs 480 crore
E. Rs 190 crore<|im_end|>
<|im_start|>assistant
Answer: B. Rs 240 crore

Explanation: 6,000 x 4.0% = 240 crore.<|im_end|>



## 7. Load the model in 4-bit and configure LoRA

In [8]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

compute_dtype = torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map={"": 0},
    attn_implementation="sdpa",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
print("Loaded:", MODEL_NAME)
print("GPU:", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct
GPU: Tesla T4


## 8. Prepare datasets and attach LoRA adapters

In [9]:
from peft import get_peft_model

train_sft = formatted["train"].remove_columns(
    [c for c in formatted["train"].column_names if c != "text"]
)
val_sft = formatted["validation"].remove_columns(
    [c for c in formatted["validation"].column_names if c != "text"]
)
test_sft = formatted["test"].remove_columns(
    [c for c in formatted["test"].column_names if c != "text"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Train:", len(train_sft), "Validation:", len(val_sft), "Test:", len(test_sft))

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
Train: 800 Validation: 100 Test: 100


## 9. Configure supervised fine-tuning

In [10]:
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM

# Qwen chat template's assistant-turn prefix. This masks prompt tokens so loss
# is calculated on assistant responses, not on system/user prompt tokens.
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    tokenizer=tokenizer,
)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
    data_seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    tokenizer=tokenizer,
    data_collator=collator,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)
print("Trainer configured. Effective batch size:",
      training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)

/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.13/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will ove

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Trainer configured. Effective batch size: 8


## 10. Verify the batch before training

In [11]:
batch = next(iter(trainer.get_train_dataloader()))
print("Batch keys:", list(batch.keys()))
print("Input shape:", tuple(batch["input_ids"].shape))
valid_labels = batch["labels"][batch["labels"] != -100]
print("Tokens contributing to loss:", valid_labels.numel())
assert valid_labels.numel() > 0, (
    "No tokens contribute to loss. Check the assistant response marker and dataset format."
)
print("Pre-training batch check passed.")

Batch keys: ['input_ids', 'attention_mask', 'labels']
Input shape: (1, 123)
Tokens contributing to loss: 27
Pre-training batch check passed.


## 11. Train and evaluate on validation set

In [12]:
import time

print("Starting QLoRA fine-tuning...")
start = time.time()
train_result = trainer.train()
elapsed_minutes = (time.time() - start) / 60

validation_metrics = trainer.evaluate()
print("Training minutes:", round(elapsed_minutes, 2))
print("Training metrics:", train_result.metrics)
print("Validation metrics:", validation_metrics)

Starting QLoRA fine-tuning...


Step,Training Loss,Validation Loss
50,0.709300,0.845122
100,0.712000,0.734355
150,0.351600,0.730559
200,0.372900,0.690156
250,0.243300,0.784212
300,0.212900,0.789456


Training minutes: 29.02
Training metrics: {'train_runtime': 1740.5488, 'train_samples_per_second': 1.379, 'train_steps_per_second': 0.172, 'total_flos': 3702631136679936.0, 'train_loss': 0.45505557735761004, 'epoch': 3.0}
Validation metrics: {'eval_loss': 0.6901564598083496, 'eval_runtime': 20.1539, 'eval_samples_per_second': 4.962, 'eval_steps_per_second': 4.962, 'epoch': 3.0}


## 12. Save adapter and actual metrics

In [13]:
ADAPTER_DIR = OUTPUT_DIR / "final_adapter"
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

metrics = {
    "base_model": MODEL_NAME,
    "method": "QLoRA (4-bit NF4) + supervised fine-tuning",
    "dataset_file": str(DATA_PATH),
    "train_examples": len(train_sft),
    "validation_examples": len(val_sft),
    "test_examples": len(test_sft),
    "epochs_configured": training_args.num_train_epochs,
    "max_sequence_length": MAX_SEQ_LENGTH,
    "training_runtime_minutes": round(elapsed_minutes, 2),
    "train_metrics": train_result.metrics,
    "validation_metrics": validation_metrics,
}
with (OUTPUT_DIR / "training_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, default=float)
print("Adapter saved to:", ADAPTER_DIR)
print("Metrics saved to:", OUTPUT_DIR / "training_metrics.json")

Adapter saved to: /content/drive/MyDrive/finexam_qlora_outputs/final_adapter
Metrics saved to: /content/drive/MyDrive/finexam_qlora_outputs/training_metrics.json


## 13. Evaluate on the held-out test split

In [14]:
# Final test evaluation is performed only after training decisions are complete.
test_eval = test_sft.map(
    lambda ex: tokenizer(ex["text"], truncation=True, max_length=MAX_SEQ_LENGTH,
                          add_special_tokens=False),
    remove_columns=["text"],
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

test_loader = DataLoader(test_eval, batch_size=1, shuffle=False, collate_fn=collator)
model.eval()
model.config.use_cache = False
total_loss, n_batches = 0.0, 0

for batch in tqdm(test_loader, desc="Test evaluation"):
    batch = {k: v.to(model.device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    if outputs.loss is not None and torch.isfinite(outputs.loss):
        total_loss += outputs.loss.item()
        n_batches += 1

test_loss = total_loss / n_batches if n_batches else float("nan")
print("Test examples evaluated:", n_batches)
print("Test loss:", round(test_loss, 4))
metrics["test_loss"] = test_loss
metrics["test_examples_evaluated"] = n_batches
with (OUTPUT_DIR / "training_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, default=float)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Test evaluation:   0%|          | 0/100 [00:00<?, ?it/s]

Test examples evaluated: 100
Test loss: 0.7793


## 14. Generate answers with the fine-tuned model

In [16]:
import torch

# Put the model in evaluation mode
model.eval()
model.config.use_cache = True

# System prompt aligned with the training dataset
SYSTEM_PROMPT = (
    "You are a careful finance-exam preparation assistant for RBI and SEBI exams. "
    "Answer multiple-choice questions by selecting the single best option from "
    "A, B, C, D, and E.\n\n"
    "For numerical questions, solve the calculation independently before choosing "
    "an option. Match the calculated result to the exact option letter and option "
    "text. Check that your final answer agrees with your explanation.\n\n"
    "Use this format:\n"
    "Answer: [LETTER]. [Exact option text]\n\n"
    "Explanation: [Explain why the answer is correct. Show calculations step by "
    "step when relevant.]\n\n"
    "Do not invent options, change the supplied values, or select an option that "
    "contradicts your explanation. Be accurate and concise."
)


def generate_answer(question, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    # Format the conversation using Qwen's chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(model.device)

    # Deterministic generation: no sampling
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=(
                tokenizer.pad_token_id
                if tokenizer.pad_token_id is not None
                else tokenizer.eos_token_id
            ),
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated answer
    input_length = inputs["input_ids"].shape[1]
    new_tokens = outputs[0, input_length:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()


# MCQ test questions matching the dataset format
test_questions = [
    """Answer this SEBI Grade A Finance multiple-choice question.
Give the best option and explain why.

A bank has Net Demand and Time Liabilities (NDTL) of Rs 6,000 crore.
At a Cash Reserve Ratio (CRR) of 4.0%, the cash maintained with RBI is:

Options:
A. Rs 340 crore
B. Rs 240 crore
C. Rs 120 crore
D. Rs 480 crore
E. Rs 190 crore""",

    """Answer this SEBI Grade A Finance multiple-choice question.
Give the best option and explain why.

A sum will grow to Rs 62,500 after 4 years at 12% compounded annually.
The amount invested today was closest to:

Options:
A. Rs 62,500
B. Rs 39,720
C. Rs 26,838
D. Rs 41,309
E. Rs 42,230""",

    """Answer this SEBI Grade A Finance multiple-choice question.
Give the best option and explain why.

A bank has total deposits of Rs 100 crore and total loans of Rs 75 crore.
What is its credit-deposit ratio?

Options:
A. 25%
B. 50%
C. 65%
D. 75%
E. 80%""",

    """Answer this SEBI Grade A Finance multiple-choice question.
Give the best option and explain why.

Which of the following best describes the primary market?

Options:
A. Investors trade existing securities with one another
B. Companies issue new securities to raise capital
C. Banks exchange foreign currencies
D. The central bank exclusively trades government securities
E. Investors receive interest on existing bonds""",

    """Answer this SEBI Grade A Finance multiple-choice question.
Give the best option and explain why.

Which of the following best describes insider trading?

Options:
A. Buying securities through a registered broker
B. Trading securities using publicly available annual reports
C. Trading using unpublished price-sensitive information in violation of applicable rules
D. Investing in an initial public offering
E. Selling securities to rebalance a portfolio"""
]


# Generate and display answers
for i, question in enumerate(test_questions, start=1):
    print("=" * 80)
    print(f"MCQ {i}")
    print("=" * 80)
    print(question)
    print("\nMODEL ANSWER:\n")

    try:
        answer = generate_answer(question)
        print(answer)
    except Exception as error:
        print(f"Error generating answer: {error}")

    print("\n")

MCQ 1
Answer this SEBI Grade A Finance multiple-choice question.
Give the best option and explain why.

A bank has Net Demand and Time Liabilities (NDTL) of Rs 6,000 crore.
At a Cash Reserve Ratio (CRR) of 4.0%, the cash maintained with RBI is:

Options:
A. Rs 340 crore
B. Rs 240 crore
C. Rs 120 crore
D. Rs 480 crore
E. Rs 190 crore

MODEL ANSWER:

Answer: B. Rs 240 crore

Explanation: Required CRR = NDTL x CRR = 6,000 x 4.0% = Rs 240 crore.


MCQ 2
Answer this SEBI Grade A Finance multiple-choice question.
Give the best option and explain why.

A sum will grow to Rs 62,500 after 4 years at 12% compounded annually.
The amount invested today was closest to:

Options:
A. Rs 62,500
B. Rs 39,720
C. Rs 26,838
D. Rs 41,309
E. Rs 42,230

MODEL ANSWER:

Answer: B. Rs 39,720

Explanation: Reversing the compounding: PV = 62,500/(1+0.12)^4 = Rs 39,720. Multiplying by the number of years ignores compounding.

Worked steps:
1. (1.12)^4 = 1.5735
2. PV = 39,720


MCQ 3
Answer this SEBI Grade A Financ

## 15. Compare base model vs fine-tuned model

In [17]:
# ============================================================
# Compare the original Qwen model with the QLoRA fine-tuned model
# ============================================================

import torch
import pandas as pd
from contextlib import nullcontext
from peft import PeftModel
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "/content/drive/MyDrive/finexam_qlora_outputs/final_adapter"

OUTPUT_PATH = (
    "/content/drive/MyDrive/finexam_qlora_outputs/"
    "base_vs_finetuned_comparison.csv"
)

SYSTEM_PROMPT = (
    "You are a careful finance-exam preparation assistant for RBI and SEBI exams. "
    "Answer multiple-choice questions by selecting the single best option from "
    "A, B, C, D, and E.\n\n"
    "For numerical questions, solve the calculation independently before choosing "
    "an option. Match the calculated result to the exact option letter and option "
    "text. Ensure your final answer agrees with your explanation.\n\n"
    "Use this format:\n"
    "Answer: [LETTER]. [Exact option text]\n\n"
    "Explanation: [Explain why the answer is correct, showing calculations when "
    "relevant. Do not invent or change options.]"
)

# These are holdout questions: do not include them in training.
mcq_questions = [
    """A deposit of Rs 20,000 earns 6% interest per year, compounded annually for 2 years. What is its maturity value?
Options:
A. Rs 22,000
B. Rs 22,400
C. Rs 22,472
D. Rs 22,600
E. Rs 24,000""",

    """A bank advertises a nominal annual rate of 12%, compounded quarterly. What is the effective annual rate, approximately?
Options:
A. 12.00%
B. 12.36%
C. 12.55%
D. 13.00%
E. 15.00%""",

    """A bond pays an annual coupon of Rs 72 and currently sells for Rs 960. What is its current yield?
Options:
A. 6.75%
B. 7.20%
C. 7.50%
D. 8.00%
E. 8.25%""",

    """A mutual fund has assets worth Rs 120 crore, liabilities of Rs 8 crore, and 14 crore units outstanding. What is its NAV per unit?
Options:
A. Rs 7
B. Rs 8
C. Rs 8.50
D. Rs 9
E. Rs 10""",

    """An investor buys a put option with a strike price of Rs 500 and pays a premium of Rs 18. At expiry, the underlying share price is Rs 460. What is the investor's net profit per share, ignoring costs?
Options:
A. Rs 18
B. Rs 22
C. Rs 40
D. Rs 42
E. A loss of Rs 18""",

    """An investor buys a 182-day Treasury bill for Rs 980 and receives Rs 1,000 at maturity. What is the holding-period return, not annualized?
Options:
A. 1.02%
B. 2.00%
C. 2.04%
D. 2.50%
E. 20.00%""",

    """In a repo transaction used to inject liquidity, which best describes the central bank's role?
Options:
A. It lends funds against securities, with an agreement to reverse the transaction later
B. It permanently buys equity shares from commercial banks
C. It raises the required cash reserve ratio
D. It guarantees a bank's unsecured corporate loans
E. It issues shares to the borrowing bank""",

    """Which type of risk is diversification across many companies most directly intended to reduce?
Options:
A. Systematic market risk
B. Inflation risk
C. Unsystematic company-specific risk
D. Interest-rate risk
E. Currency risk"""
]

# ------------------------------------------------------------
# Load tokenizer and quantized base model
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

base_model.eval()
base_model.config.use_cache = True

# ------------------------------------------------------------
# Attach the trained LoRA adapter
# ------------------------------------------------------------

ft_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH,
)

ft_model.eval()
ft_model.config.use_cache = True

# ------------------------------------------------------------
# Generate an answer with the adapter enabled or disabled
# ------------------------------------------------------------

def generate_comparison_answer(
    question,
    use_adapter,
    max_new_tokens=256,
):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(ft_model.device)

    # Disable the LoRA adapter for the baseline answer.
    adapter_context = (
        nullcontext()
        if use_adapter
        else ft_model.disable_adapter()
    )

    with adapter_context:
        with torch.inference_mode():
            output = ft_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.05,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

# ------------------------------------------------------------
# Compare both models on the same questions
# ------------------------------------------------------------

comparison_results = []

for i, question in enumerate(mcq_questions, start=1):
    print("=" * 100)
    print(f"MCQ {i}")
    print("=" * 100)
    print(question)

    print("\n[BASE MODEL ANSWER]")
    try:
        base_answer = generate_comparison_answer(
            question,
            use_adapter=False,
        )
        print(base_answer)
    except Exception as error:
        base_answer = f"ERROR: {error}"
        print(base_answer)

    print("\n[FINE-TUNED MODEL ANSWER]")
    try:
        finetuned_answer = generate_comparison_answer(
            question,
            use_adapter=True,
        )
        print(finetuned_answer)
    except Exception as error:
        finetuned_answer = f"ERROR: {error}"
        print(finetuned_answer)

    comparison_results.append({
        "MCQ": i,
        "Question": question,
        "Base Model Answer": base_answer,
        "Fine-Tuned Model Answer": finetuned_answer,
    })

    print()

# ------------------------------------------------------------
# Display and save results
# ------------------------------------------------------------

comparison_df = pd.DataFrame(comparison_results)

if not comparison_df.empty:
    display(
        comparison_df[
            ["MCQ", "Base Model Answer", "Fine-Tuned Model Answer"]
        ]
    )

    comparison_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nComparison saved to: {OUTPUT_PATH}")
else:
    print("No MCQs provided.")

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


MCQ 1
A deposit of Rs 20,000 earns 6% interest per year, compounded annually for 2 years. What is its maturity value?
Options:
A. Rs 22,000
B. Rs 22,400
C. Rs 22,472
D. Rs 22,600
E. Rs 24,000

[BASE MODEL ANSWER]
Answer: C. Rs 22,472

Explanation: The formula for compound interest is given by:

\[ A = P \left(1 + \frac{r}{n}\right)^{nt} \]

Where:
- \( A \) is the amount of money accumulated after n years, including interest.
- \( P \) is the principal amount (the initial amount of money).
- \( r \) is the annual interest rate (decimal).
- \( n \) is the number of times that interest is compounded per year.
- \( t \) is the time the money is invested for in years.

Given:
- \( P = 20,000 \)
- \( r = 0.06 \)
- \( n = 1 \) (compounded annually)
- \( t = 2 \)

Substituting these values into the formula:

\[ A = 20,000 \left(1 + \frac{0.06}{1}\right)^{1 \times 2} \]
\[ A = 20,000 \left(1 + 0.06\right)^2 \]
\[ A = 20,00

[FINE-TUNED MODEL ANSWER]
Answer: C. Rs 22,472

Explanation: Two gross

,MCQ,Base Model Answer,Fine-Tuned Model Answer
0,1,"Answer: C. Rs 22,472\n\nExplanation: The formu...","Answer: C. Rs 22,472\n\nExplanation: Two gross..."
1,2,Answer: C. 12.55%\n\nExplanation: The effectiv...,Answer: C. 12.55%\n\nExplanation: Periodic rat...
2,3,Answer: C. 7.50%\n\nExplanation: The current y...,Answer: C. 7.50%\n\nExplanation: Current yield...
3,4,Answer: C. Rs 8.50\n\nExplanation: The Net Ass...,Answer: E. Rs 10\n\nExplanation: Net assets = ...
4,5,Answer: E. A loss of Rs 18\n\nExplanation: \nT...,Answer: E. A loss of Rs 18\n\nExplanation: The...
5,6,Answer: C. 2.04%\n\nExplanation: \nThe holding...,"Answer: C. 2.04%\n\nExplanation: Gain = 1,000 ..."
6,7,"Answer: A. It lends funds against securities, ...","Answer: A. It lends funds against securities, ..."
7,8,Answer: C\n\nExplanation: Diversification acro...,Answer: C. Unsystematic company-specific risk\...



Comparison saved to: /content/drive/MyDrive/finexam_qlora_outputs/base_vs_finetuned_comparison.csv


In [18]:

import re
import os
import json
import pandas as pd
import torch
from tqdm.auto import tqdm

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
OUTPUT_DIR = "/content/drive/MyDrive/finexam_qlora_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

test_data = dataset["test"]

SYSTEM_PROMPT = (
    "You are a finance exam assistant preparing students for RBI "
    "and SEBI examinations. Answer the multiple-choice question "
    "accurately. Select exactly one option from A, B, C, D, or E. "
    "Give the final answer in the format: Answer: B"
)

# ---------------------------------------------------------
# Extract correct option from the final assistant response
# ---------------------------------------------------------
def extract_option(text):
    if not text:
        return None

    patterns = [
        r"(?im)^\s*Answer\s*:\s*[\(\[]?([A-E])[\)\]]?",
        r"(?im)^\s*(?:Correct\s+)?Option\s*:\s*[\(\[]?([A-E])[\)\]]?",
        r"(?im)^\s*Correct\s+Answer\s*:\s*[\(\[]?([A-E])[\)\]]?",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            return match.group(1).upper()

    # Fallback for answers such as "The answer is B."
    match = re.search(
        r"(?i)\b(?:the\s+)?answer\s+is\s+[\(\[]?([A-E])\b",
        text
    )
    if match:
        return match.group(1).upper()

    return None


def get_messages(example):
    # Supports the Qwen chat-message schema.
    messages = example.get("messages")
    if messages is None:
        raise ValueError("A test example does not contain 'messages'.")
    return messages


# ---------------------------------------------------------
# Evaluate one example at a time
# ---------------------------------------------------------
results = []
correct_count = 0
invalid_count = 0

model.eval()

for idx, example in enumerate(tqdm(test_data, desc="Evaluating test set")):
    messages = get_messages(example)

    # Find the final assistant response containing the answer key.
    answer_index = None
    for i in range(len(messages) - 1, -1, -1):
        if messages[i].get("role") == "assistant":
            answer_index = i
            break

    if answer_index is None:
        raise ValueError(f"Example {idx} has no assistant answer.")

    gold_text = messages[answer_index].get("content", "")
    gold_option = extract_option(gold_text)

    if gold_option is None:
        raise ValueError(
            f"Could not extract the correct option for test example {idx}.\n"
            f"Answer text: {gold_text[:500]}"
        )

    # Use all messages before the answer as the prompt.
    prompt_messages = messages[:answer_index]

    # Add the evaluation instruction if no system message exists.
    if not any(m.get("role") == "system" for m in prompt_messages):
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ] + prompt_messages

    prompt = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    # Place inputs on the device used by the model.
    input_device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(input_device) for k, v in inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
                if tokenizer.pad_token_id is not None
                else tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode only newly generated tokens.
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    prediction_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    predicted_option = extract_option(prediction_text)
    is_correct = predicted_option == gold_option

    if is_correct:
        correct_count += 1
    if predicted_option is None:
        invalid_count += 1

    # Retain useful metadata if the dataset includes it.
    row = {
        "index": idx,
        "gold_option": gold_option,
        "predicted_option": predicted_option,
        "correct": is_correct,
        "prediction_valid": predicted_option is not None,
        "gold_answer_text": gold_text,
        "model_output": prediction_text
    }

    for field in ["topic", "subject", "category", "difficulty"]:
        if field in example:
            row[field] = example[field]

    results.append(row)

    # Free per-example tensors.
    del inputs, output_ids, generated_ids


# ---------------------------------------------------------
# Summary
# ---------------------------------------------------------
results_df = pd.DataFrame(results)

total = len(results_df)
accuracy = correct_count / total if total else 0.0
valid_count = total - invalid_count

print("\n" + "=" * 55)
print("FINE-TUNED MODEL — TEST SET RESULTS")
print("=" * 55)
print(f"Test questions:       {total}")
print(f"Correct:              {correct_count}")
print(f"Incorrect:            {total - correct_count}")
print(f"Unparseable answers:  {invalid_count}")
print(f"Valid answer rate:    {valid_count / total:.2%}" if total else "")
print(f"MCQ accuracy:         {accuracy:.2%}")
print("=" * 55)

# Save every prediction, including errors.
predictions_path = os.path.join(
    OUTPUT_DIR, "finetuned_test_evaluation.csv"
)
results_df.to_csv(predictions_path, index=False)

summary = {
    "model": "Qwen/Qwen2.5-1.5B-Instruct + QLoRA adapter",
    "test_examples": total,
    "correct": correct_count,
    "incorrect": total - correct_count,
    "unparseable_predictions": invalid_count,
    "accuracy": accuracy,
    "accuracy_percent": round(accuracy * 100, 2)
}

summary_path = os.path.join(
    OUTPUT_DIR, "finetuned_test_summary.json"
)
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("\nSaved files:")
print(predictions_path)
print(summary_path)

# Show incorrect predictions for review.
errors_df = results_df[~results_df["correct"]]

print("\nFirst 10 incorrect or unparseable predictions:")
display(errors_df.head(10))

# Optional breakdown by topic or difficulty.
for field in ["topic", "subject", "category", "difficulty"]:
    if field in results_df.columns:
        print(f"\nAccuracy by {field}:")
        display(
            results_df.groupby(field, dropna=False)
            .agg(
                questions=("correct", "size"),
                correct=("correct", "sum"),
                accuracy=("correct", "mean")
            )
            .reset_index()
        )


Evaluating test set:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_


FINE-TUNED MODEL — TEST SET RESULTS
Test questions:       100
Correct:              69
Incorrect:            31
Unparseable answers:  0
Valid answer rate:    100.00%
MCQ accuracy:         69.00%

Saved files:
/content/drive/MyDrive/finexam_qlora_outputs/finetuned_test_evaluation.csv
/content/drive/MyDrive/finexam_qlora_outputs/finetuned_test_summary.json

First 10 incorrect or unparseable predictions:


,index,gold_option,predicted_option,correct,prediction_valid,gold_answer_text,model_output,topic,difficulty
1,1,B,E,False,True,Answer: B. Permitted\n\nExplanation: RTGS mini...,Answer: E. Rejected as too large\n\nExplanatio...,banking_rbi,easy
2,2,A,C,False,True,"Answer: A. Rs 84,932\n\nExplanation: 50 crore ...","Answer: C. Rs 3,10,000\n\nExplanation: Overnig...",debt_money_markets,medium
4,4,A,B,False,True,"Answer: A. I, II and III\n\nExplanation: Compe...",Answer: B. I and II only\n\nExplanation: State...,bonds,hard
5,5,D,C,False,True,"Answer: D. Rs 15,00,000\n\nExplanation: 3 x 5,...","Answer: C. Rs 1,50,000\n\nExplanation: 5,00,00...",forex,hard
6,6,A,C,False,True,"Answer: A. Rs 20,000\n\nExplanation: (86.60-86...","Answer: C. Rs 2,000\n\nExplanation: Per centag...",forex,medium
15,15,D,A,False,True,"Answer: D. -Rs 4,200\n\nExplanation: A long po...","Answer: A. +Rs 4,200\n\nExplanation: Position ...",derivatives,medium
19,19,E,D,False,True,Answer: E. 6.68%\n\nExplanation: The average a...,Answer: D. 6.86%\n\nExplanation: The yield-to-...,bonds,hard
20,20,A,C,False,True,Answer: A. 4.63%\n\nExplanation: Gain at matur...,"Answer: C. 6.02%\n\nExplanation: Gain = 75,000...",debt_money_markets,hard
22,22,B,C,False,True,Answer: B. The Central Government\n\nExplanati...,Answer: C. The Finance Commission\n\nExplanati...,banking_rbi,medium
23,23,A,C,False,True,Answer: A. 6.0 years\n\nExplanation: Doubling ...,Answer: C. 9.0 years\n\nExplanation: Doubling ...,tvm,easy



Accuracy by topic:


,topic,questions,correct,accuracy
0,banking_rbi,11,7,0.636364
1,bonds,18,15,0.833333
2,debt_money_markets,19,10,0.526316
3,derivatives,12,9,0.750000
4,equity_markets,2,0,0.000000
5,financial_institutions,2,2,1.000000
6,forex,4,2,0.500000
7,inclusion_alternate,4,4,1.000000
8,mutual_funds,3,3,1.000000
9,public_finance,10,9,0.900000



Accuracy by difficulty:


,difficulty,questions,correct,accuracy
0,easy,18,10,0.555556
1,hard,16,10,0.625000
2,medium,66,49,0.742424


In [19]:

# SECTION 17: Base model vs fine-tuned model
# Run after Section 16 has completed.

import os
import gc
import re
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "/content/drive/MyDrive/finexam_qlora_outputs"
PREDICTIONS_PATH = os.path.join(
    OUTPUT_DIR, "finetuned_test_evaluation.csv"
)

# Preserve the fine-tuned model's evaluation results.
ft_df = pd.read_csv(PREDICTIONS_PATH)
ft_accuracy = ft_df["correct"].astype(bool).mean()

# Release the fine-tuned model to reduce T4 memory pressure.
# The adapter and results are already saved on Drive.
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load a clean base model WITHOUT the LoRA adapter.
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    low_cpu_mem_usage=True
)
base_model.eval()

SYSTEM_PROMPT = (
    "You are a finance exam assistant preparing students for RBI "
    "and SEBI examinations. Answer the multiple-choice question "
    "accurately. Select exactly one option from A, B, C, D, or E. "
    "Give the final answer in the format: Answer: B"
)

def extract_mcq_option(text):
    patterns = [
        r"(?im)^\s*Answer\s*:\s*[\(\[]?([A-E])[\)\]]?",
        r"(?im)^\s*(?:Correct\s+)?Option\s*:\s*[\(\[]?([A-E])[\)\]]?",
        r"(?im)^\s*Correct\s+Answer\s*:\s*[\(\[]?([A-E])[\)\]]?",
        r"(?i)\b(?:the\s+)?answer\s+is\s+[\(\[]?([A-E])\b"
    ]
    for pattern in patterns:
        match = re.search(pattern, text or "")
        if match:
            return match.group(1).upper()
    return None

base_results = []

for idx, example in enumerate(tqdm(dataset["test"], desc="Evaluating base model")):
    messages = example["messages"]

    answer_idx = next(
        (i for i in range(len(messages) - 1, -1, -1)
         if messages[i].get("role") == "assistant"),
        None
    )
    if answer_idx is None:
        raise ValueError(f"No assistant answer found for example {idx}")

    gold = extract_mcq_option(messages[answer_idx].get("content", ""))
    if gold is None:
        raise ValueError(f"Could not parse gold answer for example {idx}")

    prompt_messages = messages[:answer_idx]

    if not any(m.get("role") == "system" for m in prompt_messages):
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT}
        ] + prompt_messages

    prompt = base_tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = base_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    device = base_model.get_input_embeddings().weight.device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        output = base_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=base_tokenizer.pad_token_id
                if base_tokenizer.pad_token_id is not None
                else base_tokenizer.eos_token_id,
            eos_token_id=base_tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer_text = base_tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    pred = extract_mcq_option(answer_text)

    base_results.append({
        "index": idx,
        "gold_option": gold,
        "predicted_option": pred,
        "correct": pred == gold,
        "model_output": answer_text
    })

    del inputs, output, new_tokens

base_df = pd.DataFrame(base_results)
base_accuracy = base_df["correct"].mean()

base_csv = os.path.join(OUTPUT_DIR, "base_test_evaluation.csv")
base_df.to_csv(base_csv, index=False)

comparison = pd.DataFrame([
    {
        "model": "Base Qwen2.5-1.5B-Instruct",
        "test_questions": len(base_df),
        "correct": int(base_df["correct"].sum()),
        "accuracy_percent": round(base_accuracy * 100, 2)
    },
    {
        "model": "QLoRA fine-tuned model",
        "test_questions": len(ft_df),
        "correct": int(ft_df["correct"].astype(bool).sum()),
        "accuracy_percent": round(ft_accuracy * 100, 2)
    }
])

comparison["difference_vs_base_percentage_points"] = [
    0.0,
    round((ft_accuracy - base_accuracy) * 100, 2)
]

comparison_path = os.path.join(OUTPUT_DIR, "base_vs_finetuned_comparison.csv")
comparison.to_csv(comparison_path, index=False)

print("\nBASE VS FINE-TUNED MODEL")
display(comparison)
print("\nSaved:", base_csv)
print("Saved:", comparison_path)

# Release base model memory after evaluation.
del base_model, base_tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Evaluating base model:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(



BASE VS FINE-TUNED MODEL


,model,test_questions,correct,accuracy_percent,difference_vs_base_percentage_points
0,Base Qwen2.5-1.5B-Instruct,100,37,37.0,0.0
1,QLoRA fine-tuned model,100,69,69.0,32.0



Saved: /content/drive/MyDrive/finexam_qlora_outputs/base_test_evaluation.csv
Saved: /content/drive/MyDrive/finexam_qlora_outputs/base_vs_finetuned_comparison.csv


In [20]:
import re
import pandas as pd

# Requires the existing function:
# generate_comparison_answer(question, use_adapter, max_new_tokens=...)
# use_adapter=False -> base model
# use_adapter=True  -> fine-tuned model

mcq_questions = [
    """A deposit of Rs 20,000 earns 5% interest per year, compounded annually for 3 years. What is its maturity value?
Options:
A. Rs 23,100
B. Rs 23,000
C. Rs 23,152.50
D. Rs 23,500
E. Rs 24,000""",

    """What is the present value of Rs 50,000 receivable in 2 years if the annual discount rate is 8%, compounded annually?
Options:
A. Rs 42,866.94
B. Rs 44,000
C. Rs 46,296.30
D. Rs 40,000
E. Rs 46,000""",

    """A nominal annual interest rate of 10% is compounded semi-annually. What is the effective annual rate?
Options:
A. 10.00%
B. 10.10%
C. 10.50%
D. 10.25%
E. 11.00%""",

    """A principal of Rs 12,000 earns simple interest at 7.5% per year for 2 years. What is the total amount at the end?
Options:
A. Rs 13,500
B. Rs 13,600
C. Rs 13,750
D. Rs 14,000
E. Rs 13,800""",

    """A bond pays an annual coupon of Rs 84 and is priced at Rs 1,050. What is its current yield?
Options:
A. 7.00%
B. 8.00%
C. 8.25%
D. 9.00%
E. 10.00%""",

    """An investor buys a security for Rs 920 and receives Rs 1,000 at maturity. What is the holding-period return, not annualized?
Options:
A. 8.00%
B. 8.25%
C. 8.70%
D. 9.20%
E. 10.00%""",

    """A mutual fund has assets of Rs 76 crore, liabilities of Rs 4 crore, and 8 crore units outstanding. What is its NAV per unit?
Options:
A. Rs 9
B. Rs 8
C. Rs 9.50
D. Rs 10
E. Rs 12""",

    """A share is priced at Rs 315 and its earnings per share are Rs 21. What is its price-to-earnings ratio?
Options:
A. 12
B. 14
C. 14.5
D. 15
E. 16""",

    """A company has 18 million shares outstanding, each trading at Rs 125. What is its market capitalization?
Options:
A. Rs 18 crore
B. Rs 125 crore
C. Rs 180 crore
D. Rs 2,250 crore
E. Rs 225 crore""",

    """An investor buys a put option with a strike price of Rs 400 and pays a premium of Rs 14. At expiry, the share price is Rs 365. What is the net profit per share, ignoring costs?
Options:
A. Rs 14
B. Rs 21
C. Rs 35
D. Rs 49
E. A loss of Rs 14""",

    """An investor buys a call option with a strike price of Rs 150 and pays a premium of Rs 8. At expiry, the share price is Rs 170. What is the net profit per share, ignoring costs?
Options:
A. Rs 8
B. Rs 10
C. Rs 12
D. Rs 20
E. Rs 28""",

    """A trader sells a futures contract at Rs 2,250. At settlement, the futures price is Rs 2,190. What is the trader's profit per unit, ignoring costs?
Options:
A. Rs 60 profit
B. Rs 60 loss
C. Rs 2,190 profit
D. Rs 2,250 loss
E. Rs 120 profit""",

    """A project costs Rs 5,000 today and is expected to generate Rs 3,000 after one year and Rs 3,000 after two years. At a 10% annual discount rate, what is its NPV, approximately?
Options:
A. -Rs 500
B. -Rs 206.61
C. Rs 0
D. Rs 206.61
E. Rs 500""",

    """A bank has net demand and time liabilities of Rs 4,250 crore. At a CRR of 4%, as stated in the question, what amount does this ratio imply?
Options:
A. Rs 85 crore
B. Rs 127.50 crore
C. Rs 150 crore
D. Rs 200 crore
E. Rs 170 crore""",

    """A bank has eligible capital of Rs 720 crore and risk-weighted assets of Rs 8,000 crore. What is its capital-to-risk-weighted-assets ratio?
Options:
A. 8%
B. 9%
C. 9.5%
D. 10%
E. 11%""",

    """The exchange rate changes from Rs 83 per US dollar to Rs 81 per US dollar. What does this indicate, all else equal?
Options:
A. The rupee appreciated against the dollar
B. The rupee depreciated against the dollar
C. The dollar appreciated against the rupee
D. Both currencies depreciated
E. The exchange rate did not change""",

    """A bank borrows funds through a repo transaction. Which description best fits the transaction from the bank's perspective?
Options:
A. It permanently sells equity to the lender
B. It borrows without providing any security
C. It sells securities and agrees to repurchase them later
D. It issues new shares to the central bank
E. It converts deposits into equity""",

    """For the buyer of a plain-vanilla call option, what is the maximum loss, ignoring transaction costs?
Options:
A. Unlimited
B. The strike price
C. The premium paid
D. The underlying asset's market price
E. The difference between strike and spot price at expiry""",

    """Holding other factors constant, what is the usual relationship between a bond's market yield and its price?
Options:
A. They always rise together
B. They are unrelated
C. Both remain fixed
D. They generally move in opposite directions
E. Price changes only when the coupon changes""",

    """What typically happens in a secondary securities market?
Options:
A. A company issues shares for the first time
B. Investors trade securities that have already been issued
C. The government collects income tax
D. A bank calculates its capital ratio
E. A company declares a dividend""",

    """Which description best fits money-market instruments?
Options:
A. Short-term debt instruments, commonly with maturities up to about one year
B. Permanent ownership claims in companies
C. Only long-term government bonds
D. Perpetual securities with no maturity
E. Residential mortgage loans""",

    """Diversifying a portfolio across many unrelated companies most directly reduces which risk?
Options:
A. Systematic market risk
B. Inflation risk
C. Unsystematic company-specific risk
D. Economy-wide interest-rate risk
E. General purchasing-power risk""",

    """If a project's NPV is positive using an appropriate required rate of return, what is the standard capital-budgeting implication?
Options:
A. Reject it because all future cash flows are uncertain
B. Accept it only if it pays back within one year
C. Ignore the discount rate
D. The project has zero economic value
E. Accept it because discounted expected inflows exceed the investment""",

    """Which statement best describes a direct tax?
Options:
A. It is collected only by stock exchanges
B. It is imposed directly on the income or wealth of the person or entity liable under the law
C. Its burden is always shifted entirely to consumers
D. It applies only to imported goods
E. It is a fee for a specific public service""",

    """How does a mutual fund's recurring expense ratio generally affect investors?
Options:
A. It is paid by the government and increases NAV
B. It guarantees a minimum return
C. It changes the number of units held each day
D. It is charged to scheme assets and reduces returns available to investors
E. It applies only when units are bought on an exchange""",

    """An exporter expects to receive US dollars in three months and wants to fix the rupee amount it will receive. Which action most directly locks in an exchange rate?
Options:
A. Sell the expected US dollars forward
B. Buy more US dollars immediately
C. Borrow in rupees and leave the future receipt unhedged
D. Buy an unrelated company share
E. Hold the currency exposure without a contract""",

    """A bank's interest income is Rs 120 crore, its interest expense is Rs 70 crore, and its average interest-earning assets are Rs 1,000 crore. What is its net interest margin?
Options:
A. 4.0%
B. 4.5%
C. 5.0%
D. 6.0%
E. 7.0%""",

    """An investor wants to reduce the risk that a single company's poor performance will dominate their portfolio. Which approach most directly addresses this?
Options:
A. Invest all funds in that company's shares
B. Diversify across different companies
C. Borrow more to increase the position
D. Hold only cash in one currency
E. Concentrate in one industry""",

    """In the approximate Fisher relationship, the nominal interest rate is approximately equal to which combination?
Options:
A. Real interest rate minus expected inflation
B. Expected inflation minus the real interest rate
C. Real interest rate multiplied by expected inflation
D. Real interest rate plus expected inflation
E. Expected inflation divided by the real interest rate""",

    """Compared with an otherwise similar unleveraged investment, what does financial leverage generally do to returns on the investor's own equity?
Options:
A. Eliminates both gains and losses
B. Guarantees a higher return
C. Converts all risk into systematic risk
D. Affects only the amount of tax paid
E. Magnifies gains and losses"""
]

# Correct answer key for scoring only. Do not add it to model prompts.
answer_key = {
    1: "C",  2: "A",  3: "D",  4: "E",  5: "B",
    6: "C",  7: "A",  8: "D",  9: "E", 10: "B",
    11: "C", 12: "A", 13: "D", 14: "E", 15: "B",
    16: "A", 17: "C", 18: "C", 19: "D", 20: "B",
    21: "A", 22: "C", 23: "E", 24: "B", 25: "D",
    26: "A", 27: "C", 28: "B", 29: "D", 30: "E",
}

def extract_answer_letter(text):
    match = re.search(r"(?im)^\s*Answer\s*:\s*([A-E])\b", text)
    return match.group(1).upper() if match else None

results = []

for i, question in enumerate(mcq_questions, start=1):
    base_text = generate_comparison_answer(
        question, use_adapter=False, max_new_tokens=350
    )
    ft_text = generate_comparison_answer(
        question, use_adapter=True, max_new_tokens=350
    )

    base_choice = extract_answer_letter(base_text)
    ft_choice = extract_answer_letter(ft_text)
    correct = answer_key[i]

    results.append({
        "Question": i,
        "Correct": correct,
        "Base choice": base_choice,
        "Base correct": base_choice == correct,
        "Fine-tuned choice": ft_choice,
        "Fine-tuned correct": ft_choice == correct,
        "Base response": base_text,
        "Fine-tuned response": ft_text,
    })

results_df = pd.DataFrame(results)

print("Base accuracy:",
      f"{results_df['Base correct'].sum()}/{len(results_df)}")
print("Fine-tuned accuracy:",
      f"{results_df['Fine-tuned correct'].sum()}/{len(results_df)}")

display(results_df[
    ["Question", "Correct", "Base choice", "Base correct",
     "Fine-tuned choice", "Fine-tuned correct"]
])

# Uncomment to inspect full explanations:
# display(results_df[["Question", "Base response", "Fine-tuned response"]])

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:464: FutureWarning: _check_

Base accuracy: 23/30
Fine-tuned accuracy: 20/30


,Question,Correct,Base choice,Base correct,Fine-tuned choice,Fine-tuned correct
0,1,C,None,False,C,True
1,2,A,A,True,C,False
2,3,D,D,True,D,True
3,4,E,E,True,E,True
4,5,B,B,True,B,True
5,6,C,C,True,B,False
6,7,A,A,True,C,False
7,8,D,D,True,D,True
8,9,E,D,False,E,True
9,10,B,None,False,B,True


In [21]:
import torch

if torch.cuda.is_available():
    torch.cuda.synchronize()

    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Total GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GiB"
    )
    print(
        "Currently allocated:",
        round(torch.cuda.memory_allocated(0) / 1024**3, 3),
        "GiB"
    )
    print(
        "Currently reserved:",
        round(torch.cuda.memory_reserved(0) / 1024**3, 3),
        "GiB"
    )
    print(
        "Peak allocated:",
        round(torch.cuda.max_memory_allocated(0) / 1024**3, 3),
        "GiB"
    )
    print(
        "Peak reserved:",
        round(torch.cuda.max_memory_reserved(0) / 1024**3, 3),
        "GiB"
    )

GPU: Tesla T4
Total GPU memory: 14.56 GiB
Currently allocated: 2.834 GiB
Currently reserved: 2.922 GiB
Peak allocated: 5.729 GiB
Peak reserved: 5.93 GiB


In [22]:
import os

adapter_dir = "/content/drive/MyDrive/finexam_qlora_outputs/final_adapter"

total_bytes = sum(
    os.path.getsize(os.path.join(root, filename))
    for root, _, files in os.walk(adapter_dir)
    for filename in files
)

print(f"Adapter folder size: {total_bytes / 1024**2:.2f} MiB")

Adapter folder size: 85.64 MiB
